# Oracle Cloud DB — Administration Notebook

Operations covered:
- **Connect** to Oracle Cloud DB
- **List** all tables (user tables, vector tables, knowledge graph tables)
- **Describe** a table (columns, types, constraints)
- **Query** any table with custom SQL
- **Create** a new table
- **Modify** a table (add / drop columns)
- **Truncate** a table (delete all rows, keep structure)
- **Drop** a table
- **Inspect vector tables** (`SYSTEM_DESIGN_VECTORS`, `SYSTEM_DESIGN_GRAPH`, `SOLUTION_ARCHITECT_VECTORS`, `DOCUMENT_VECTORS`)
- **Browse knowledge graph** entries
- **Delete specific rows** by condition

**Prerequisites:** env vars `ORACLE_USER`, `ORACLE_PASS`, `ORACLE_TLS_DNS`

## 1. Install dependencies (run once)

In [ ]:
# !pip install oracledb pandas tabulate

## 2. Connect

In [1]:
import os
import oracledb
import pandas as pd
from tabulate import tabulate



def run_query(sql: str, params=None, fetch: bool = True) -> pd.DataFrame | None:
    """Execute SQL. Returns a DataFrame for SELECT, None for DDL/DML."""
    with connection.cursor() as cur:
        cur.execute(sql, params or [])
        if fetch and cur.description:
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
            return pd.DataFrame(rows, columns=cols)
        connection.commit()
        return None

def show(df: pd.DataFrame, max_rows: int = 50):
    """Pretty-print a DataFrame."""
    if df is None or df.empty:
        print("(no rows)")
        return
    print(tabulate(df.head(max_rows), headers="keys", tablefmt="rounded_outline", showindex=False))
    if len(df) > max_rows:
        print(f"  ... {len(df) - max_rows} more rows")



Connected to Oracle DB


In [10]:
connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASS"],
    dsn=os.environ["ORACLE_TLS_DNS"]
)
print("Connected to Oracle DB")

Connected to Oracle DB


## 3. List all user tables

In [2]:
df = run_query("""
    SELECT table_name, num_rows, last_analyzed
    FROM user_tables
    ORDER BY table_name
""")
show(df)

╭───────────────────────────────────────────────────────────────────────────────────┬────────────┬─────────────────────╮
│ TABLE_NAME                                                                        │   NUM_ROWS │ LAST_ANALYZED       │
├───────────────────────────────────────────────────────────────────────────────────┼────────────┼─────────────────────┤
│ DBTOOLS$EXECUTION_HISTORY                                                         │        nan │ NaT                 │
│ DOCUMENT_VECTORS                                                                  │          2 │ 2026-09-12 05:16:06 │
│ SOLUTION_ARCHITECT_VECTORS                                                        │         16 │ 2026-09-12 06:18:37 │
│ SYSTEM_DESIGN_GRAPH                                                               │        nan │ NaT                 │
│ SYSTEM_DESIGN_VECTORS                                                             │        nan │ NaT                 │
│ VECTOR$DOCUMENT_HNSW_IDX$14946

## 4. Describe a table

In [3]:
# ── Change TABLE_NAME to the table you want to inspect ────────────────────
TABLE_NAME = "SYSTEM_DESIGN_VECTORS"

df_cols = run_query("""
    SELECT column_name, data_type, data_length, data_precision, nullable
    FROM user_tab_columns
    WHERE table_name = :tname
    ORDER BY column_id
""", params={"tname": TABLE_NAME.upper()})

print(f"Columns in {TABLE_NAME}:")
show(df_cols)

# Primary key / constraints
df_pk = run_query("""
    SELECT c.constraint_name, c.constraint_type, cc.column_name
    FROM user_constraints c
    JOIN user_cons_columns cc ON c.constraint_name = cc.constraint_name
    WHERE c.table_name = :tname
    ORDER BY c.constraint_type, cc.position
""", params={"tname": TABLE_NAME.upper()})

print(f"\nConstraints in {TABLE_NAME}:")
show(df_pk)

Columns in SYSTEM_DESIGN_VECTORS:
╭───────────────┬─────────────┬───────────────┬──────────────────┬────────────╮
│ COLUMN_NAME   │ DATA_TYPE   │   DATA_LENGTH │ DATA_PRECISION   │ NULLABLE   │
├───────────────┼─────────────┼───────────────┼──────────────────┼────────────┤
│ ID            │ RAW         │            16 │                  │ N          │
│ TEXT          │ CLOB        │          4000 │                  │ Y          │
│ METADATA      │ JSON        │          8200 │                  │ Y          │
│ EMBEDDING     │ VECTOR      │          8200 │                  │ Y          │
╰───────────────┴─────────────┴───────────────┴──────────────────┴────────────╯

Constraints in SYSTEM_DESIGN_VECTORS:
╭───────────────────┬───────────────────┬───────────────╮
│ CONSTRAINT_NAME   │ CONSTRAINT_TYPE   │ COLUMN_NAME   │
├───────────────────┼───────────────────┼───────────────┤
│ SYS_C0036504      │ P                 │ ID            │
╰───────────────────┴───────────────────┴──────────────

## 5. Query any table

In [4]:
# ── Edit the SQL below ────────────────────────────────────────────────────
sql = """
    SELECT *
    FROM SYSTEM_DESIGN_VECTORS
    FETCH FIRST 10 ROWS ONLY
"""

df = run_query(sql)
show(df)

(no rows)


## 6. Create a new table

In [ ]:
# ── Edit the DDL below ────────────────────────────────────────────────────
create_sql = """
    CREATE TABLE MY_NEW_TABLE (
        id          NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        name        VARCHAR2(255)  NOT NULL,
        description CLOB,
        created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
"""

run_query(create_sql, fetch=False)
print("Table MY_NEW_TABLE created")

## 7. Add a column to an existing table

In [ ]:
# ── Edit table and column ─────────────────────────────────────────────────
alter_sql = "ALTER TABLE MY_NEW_TABLE ADD (status VARCHAR2(50) DEFAULT 'active')"

run_query(alter_sql, fetch=False)
print("Column added")

## 8. Drop a column

In [ ]:
# ── Edit table and column ─────────────────────────────────────────────────
drop_col_sql = "ALTER TABLE MY_NEW_TABLE DROP COLUMN status"

run_query(drop_col_sql, fetch=False)
print("Column dropped")

## 9. Insert rows

In [ ]:
# ── Edit insert SQL ───────────────────────────────────────────────────────
insert_sql = """
    INSERT INTO MY_NEW_TABLE (name, description)
    VALUES (:name, :description)
"""

run_query(insert_sql, params={"name": "test-entry", "description": "Sample row"}, fetch=False)
print("Row inserted")

## 10. Delete specific rows

In [ ]:
# ── Edit the WHERE condition ──────────────────────────────────────────────
delete_sql = "DELETE FROM MY_NEW_TABLE WHERE name = :name"

with connection.cursor() as cur:
    cur.execute(delete_sql, {"name": "test-entry"})
    print(f"{cur.rowcount} row(s) deleted")
    connection.commit()

## 11. Truncate a table  ⚠️ deletes ALL rows

In [ ]:
# ── Change TABLE_NAME — this deletes ALL rows ─────────────────────────────
TRUNCATE_TABLE = "MY_NEW_TABLE"

confirm = input(f"Type YES to truncate {TRUNCATE_TABLE}: ")
if confirm == "YES":
    run_query(f"TRUNCATE TABLE {TRUNCATE_TABLE}", fetch=False)
    print(f"{TRUNCATE_TABLE} truncated")
else:
    print("Cancelled")

## 12. Drop a table  ⚠️ permanent

In [ ]:
# ── Change DROP_TABLE — this permanently deletes the table ────────────────
DROP_TABLE = "MY_NEW_TABLE"

confirm = input(f"Type YES to drop {DROP_TABLE}: ")
if confirm == "YES":
    run_query(f"DROP TABLE {DROP_TABLE} PURGE", fetch=False)
    print(f"{DROP_TABLE} dropped")
else:
    print("Cancelled")

---
## 13. Vector table inspection

Inspect the RAG and knowledge graph vector tables used across notebooks.

In [ ]:
VECTOR_TABLES = [
    "DOCUMENT_VECTORS",
    "SOLUTION_ARCHITECT_VECTORS",
    "SYSTEM_DESIGN_VECTORS",
    "SYSTEM_DESIGN_GRAPH",
]

for tbl in VECTOR_TABLES:
    try:
        df = run_query(f"SELECT COUNT(*) AS row_count FROM {tbl}")
        count = df["ROW_COUNT"].iloc[0] if df is not None and not df.empty else 0
        print(f"  {tbl:<40} {count:>6} rows")
    except Exception as e:
        print(f"  {tbl:<40} ERROR: {e}")

## 14. Browse vector table — view stored chunks

In [11]:
# ── Change to any vector table ────────────────────────────────────────────
VEC_TABLE = "SYSTEM_DESIGN_VECTORS"
LIMIT = 10

# Get column names first
df_cols = run_query("""
    SELECT column_name FROM user_tab_columns
    WHERE table_name = :tname
    ORDER BY column_id
""", params={"tname": VEC_TABLE.upper()})

print(f"Columns in {VEC_TABLE}: {df_cols['COLUMN_NAME'].tolist()}")

# Browse non-vector columns (skip BLOB/vector columns for readability)
readable_cols = [
    c for c in df_cols["COLUMN_NAME"].tolist()
    if c not in ("EMBEDDING", "VECTOR") and not c.endswith("_VECTOR")
]

cols_sql = ", ".join(readable_cols) if readable_cols else "*"
df = run_query(f"SELECT {cols_sql} FROM {VEC_TABLE} FETCH FIRST {LIMIT} ROWS ONLY")
show(df)

Columns in SYSTEM_DESIGN_VECTORS: ['ID', 'TEXT', 'METADATA', 'EMBEDDING']
╭──────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ID                               │ TEXT                                                                                                                                                                                                                                                                                                           │ METADATA                                                                    

In [8]:
# ── Change to any vector table ────────────────────────────────────────────
VEC_TABLE = "SOLUTION_ARCHITECT_VECTORS"
LIMIT = 10

# Get column names first
df_cols = run_query("""
    SELECT column_name FROM user_tab_columns
    WHERE table_name = :tname
    ORDER BY column_id
""", params={"tname": VEC_TABLE.upper()})

print(f"Columns in {VEC_TABLE}: {df_cols['COLUMN_NAME'].tolist()}")

# Browse non-vector columns (skip BLOB/vector columns for readability)
readable_cols = [
    c for c in df_cols["COLUMN_NAME"].tolist()
    if c not in ("EMBEDDING", "VECTOR") and not c.endswith("_VECTOR")
]

cols_sql = ", ".join(readable_cols) if readable_cols else "*"
df = run_query(f"SELECT {cols_sql} FROM {VEC_TABLE} FETCH FIRST {LIMIT} ROWS ONLY")
show(df)

Columns in SOLUTION_ARCHITECT_VECTORS: ['ID', 'TEXT', 'METADATA', 'EMBEDDING']
╭──────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ID                               │ TEXT                                                                                                                                                                                                                                          

## 15. Search vector table by text (metadata / content filter)

In [ ]:
# ── Filter rows by source metadata ───────────────────────────────────────
VEC_TABLE = "SYSTEM_DESIGN_VECTORS"
SOURCE_FILTER = "%inventory%"   # uses SQL LIKE — change to filter by source name

df = run_query(f"""
    SELECT id, metadata, SUBSTR(text, 1, 300) AS text_preview
    FROM {VEC_TABLE}
    WHERE LOWER(TO_CHAR(metadata)) LIKE LOWER(:src)
    FETCH FIRST 10 ROWS ONLY
""", params={"src": SOURCE_FILTER})

show(df)

## 16. Browse knowledge graph (SYSTEM_DESIGN_GRAPH)

In [13]:
KG_TABLE = "SYSTEM_DESIGN_GRAPH"

# Show all component entries (text column contains component name/type/description)
df = run_query(f"""
    SELECT id, metadata, SUBSTR(text, 1, 50) AS component_text
    FROM {KG_TABLE}
    FETCH FIRST 20 ROWS ONLY
""")

print(f"Knowledge graph entries in {KG_TABLE}:")
show(df)

Knowledge graph entries in SYSTEM_DESIGN_GRAPH:
╭─────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────╮
│ ID                          │ METADATA                                                                                                                                                                                               │ COMPONENT_TEXT                                     │
├─────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────────────────────────┤
│ b'\xe1O&\x10\x90\x872\xd2'  │ {'name': 'Model Retrieval', 'type': 'Microservice', 'relations

In [14]:
df

,ID,METADATA,COMPONENT_TEXT
0,b'\xe1O&\x10\x90\x872\xd2',"{'name': 'Model Retrieval', 'type': 'Microserv...",Component: Model Retrieval\nType: Microservice...
1,b'Tg=\xecy\xc5\xaee',"{'name': 'Database', 'type': 'Relational Datab...",Component: Database\nType: Relational Database...
2,b'\xac_\xc8\xbe\x9ab~\xbd',"{'name': 'API Gateway', 'type': 'Serverless Fu...",Component: API Gateway\nType: Serverless Funct...
3,b'\x1a/\xb1\xbe\xc6S\xc1a',{'name': 'Natural Language Processing (NLP) Co...,Component: Natural Language Processing (NLP) C...
4,b'\nA\x04!\x01\xc0\xf3\xcd',"{'name': 'Search Engine', 'type': 'Database In...",Component: Search Engine\nType: Database Index...


## 17. Delete rows from a vector table by source

In [ ]:
# ── Deletes all chunks with a specific source from a vector table ─────────
VEC_TABLE    = "SYSTEM_DESIGN_VECTORS"
SOURCE_NAME  = "inventory-search-v1"   # exact source identifier used at ingest time

confirm = input(f"Type YES to delete all rows with source '{SOURCE_NAME}' from {VEC_TABLE}: ")
if confirm == "YES":
    with connection.cursor() as cur:
        cur.execute(f"""
            DELETE FROM {VEC_TABLE}
            WHERE LOWER(TO_CHAR(metadata)) LIKE :src
        """, {"src": f"%{SOURCE_NAME.lower()}%"})
        print(f"{cur.rowcount} row(s) deleted")
        connection.commit()
else:
    print("Cancelled")

## 18. Clear an entire vector table  ⚠️ deletes ALL vectors

In [ ]:
# ── Clears ALL rows from a vector / knowledge graph table ─────────────────
CLEAR_TABLE = "SYSTEM_DESIGN_GRAPH"   # change to target table

confirm = input(f"Type YES to TRUNCATE (clear all rows from) {CLEAR_TABLE}: ")
if confirm == "YES":
    run_query(f"TRUNCATE TABLE {CLEAR_TABLE}", fetch=False)
    print(f"{CLEAR_TABLE} cleared")
else:
    print("Cancelled")

## 19. Table row count summary

In [ ]:
df_all = run_query("SELECT table_name FROM user_tables ORDER BY table_name")

summary = []
for tbl in df_all["TABLE_NAME"].tolist():
    try:
        df_cnt = run_query(f"SELECT COUNT(*) AS cnt FROM {tbl}")
        cnt = int(df_cnt["CNT"].iloc[0]) if df_cnt is not None else 0
    except Exception:
        cnt = -1
    summary.append({"TABLE": tbl, "ROWS": cnt})

show(pd.DataFrame(summary))

## 20. Close connection

In [ ]:
connection.close()
print("Connection closed")